# DizerCore LoRA Training — auto-matches installed model

Train a TrinityCore-specific LoRA adapter on Kaggle's free T4/P100 GPU.

**Setup:**
1. Attach your Kaggle dataset containing BOTH files:
   - `dizercore-dataset.jsonl` (required)
   - `dizercore-base.txt` (optional — selects the base model; falls back to 1.5B-Instruct)
2. Settings → Accelerator → T4 GPU
3. Run All

Runtime: ~2 hours on T4 (1.5B). 3B is slower and needs more disk.

In [ ]:
import subprocess, sys, os, gc

print('=== Install ===', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'transformers', 'peft', 'bitsandbytes',
                'accelerate', 'datasets', 'hf_transfer'], check=True)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# ---------- Config ----------
BASE_FILE = '/kaggle/input/dizercore-dataset/dizercore-base.txt'
DATASET = '/kaggle/input/dizercore-dataset/dizercore-dataset.jsonl'
try:
    BASE = open(BASE_FILE).read().strip()
except Exception:
    BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
print('Base model:', BASE)
assert os.path.isfile(DATASET), f'Dataset not found at {DATASET}'
WORK = '/kaggle/working/dizercore'
ADAPTER = f'{WORK}/adapter'
MERGED = f'{WORK}/merged'
GGUF_F16 = f'{WORK}/dizercore-f16.gguf'
GGUF_Q4 = f'{WORK}/dizercore-q4_k_m.gguf'
os.makedirs(WORK, exist_ok=True)

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

print('=== Load dataset ===', flush=True)
ds = load_dataset('json', data_files=DATASET, split='train')
print(f'{len(ds)} examples', flush=True)

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token

def fmt(ex):
    msgs = [{'role': 'system', 'content': 'You are a TrinityCore codebase assistant.'}]
    msgs += ex['messages']
    text = tok.apply_chat_template(msgs, tokenize=False)
    out = tok(text, truncation=True, max_length=1024)
    out['labels'] = out['input_ids'].copy()
    return out

ds = ds.map(fmt, remove_columns=ds.column_names)

print('=== Load model (4-bit) ===', flush=True)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

print('=== Train ===', flush=True)
args = TrainingArguments(output_dir=f'{WORK}/ckpt', per_device_train_batch_size=4,
                         gradient_accumulation_steps=4, num_train_epochs=1, learning_rate=2e-4,
                         fp16=True, logging_steps=20, save_only_model=True, save_total_limit=1,
                         report_to=[], seed=42)
trainer = Trainer(model=model, args=args, train_dataset=ds)
trainer.train()
model.save_pretrained(ADAPTER)
tok.save_pretrained(ADAPTER)
print('Adapter saved', flush=True)

print('=== Merge LoRA ===', flush=True)
del model, trainer
gc.collect()
torch.cuda.empty_cache()
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='cpu')
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
merged.save_pretrained(MERGED)
tok.save_pretrained(MERGED)
del base, merged
gc.collect()
print('Merged model saved', flush=True)

print('=== Convert to GGUF ===', flush=True)
LLAMA_DIR = '/kaggle/working/llama.cpp'
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', LLAMA_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{LLAMA_DIR}/requirements.txt'], check=True)
subprocess.run([sys.executable, f'{LLAMA_DIR}/convert_hf_to_gguf.py', MERGED, '--outfile', GGUF_F16, '--outtype', 'f16'], check=True)

print('=== Build llama-quantize ===', flush=True)
subprocess.run(['cmake', '-B', f'{LLAMA_DIR}/build', '-S', LLAMA_DIR, '-DGGML_CUDA=OFF'], check=True)
subprocess.run(['cmake', '--build', f'{LLAMA_DIR}/build', '--target', 'llama-quantize', '-j', '4'], check=True)

print('=== Quantizing to Q4_K_M ===', flush=True)
subprocess.run([f'{LLAMA_DIR}/build/bin/llama-quantize', GGUF_F16, GGUF_Q4, 'Q4_K_M'], check=True)

size_mb = os.path.getsize(GGUF_Q4) / 1024 / 1024
print('=== DONE ===', flush=True)
print(f'GGUF: {size_mb:.0f} MB at {GGUF_Q4}', flush=True)
print('Download it from the Output tab after the run finishes, then upload via the Training tab on the Pi.', flush=True)